# SSSL-Based Continual IDS on AnoShift (NeurIPS 2022 Benchmark)
### VS Code + Google Colab Remote Kernel Guide

This notebook trains and evaluates the **Self-Supervised Semi-Supervised Learning (SSSL)** Intrusion Detection System with **Gradient Projection Memory (GPM)** on the **AnoShift** longitudinal distribution shift benchmark.

### Supported Continual Task Heads:
1. **Task 1: General Intrusion Head** (`--task intrusion`) &mdash; Malicious flow detection vs benign traffic.
2. **Task 2: Denial of Service (DoS) Head** (`--task dos`) &mdash; SYN floods, connection floods, and error bursts.
3. **Task 3: Port Scan Head** (`--task port_scan`) &mdash; Reconnaissance, host sweeps, and probing.
4. **Zero-Day Honeypot Detector** (`evaluate_zeroday.py`) &mdash; Autoencoder catches unknown honeypot behavioral anomalies (Kyoto label `-2`).

## Step 1: Environment Setup & Sync Repository on Colab VM

In [1]:
import os, sys
import tensorflow as tf

print("Python Executable:", sys.executable)
print("TensorFlow Version:", tf.__version__)
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

# Automatic Colab VM Workspace Sync
if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = '/content/sssl-continual-ids'
    if not os.path.exists(repo_dir):
        print("\n[INFO] Cloning sssl-continual-ids repository into Colab VM...")
        !git clone -b Experimental_Analysis https://github.com/yaswanthvuppala/sssl-continual-ids.git /content/sssl-continual-ids
    else:
        print("\n[INFO] Syncing latest updates from GitHub...")
        !cd /content/sssl-continual-ids && git fetch origin Experimental_Analysis && git reset --hard origin/Experimental_Analysis
    
    %cd /content/sssl-continual-ids/ids-system
else:
    # Local VS Code environment fallback
    if os.path.exists('ids-system'):
        %cd ids-system
    elif os.path.basename(os.getcwd()) != 'ids-system' and os.path.exists('../ids-system'):
        %cd ../ids-system

print("\nActive Working Directory:", os.getcwd())
!git log -1 --oneline

Python Executable: /usr/bin/python3
TensorFlow Version: 2.20.0
GPU Devices Available: []

[INFO] Cloning sssl-continual-ids repository into Colab VM...
Cloning into '/content/sssl-continual-ids'...
remote: Enumerating objects: 461, done.
remote: Counting objects: 100% (461/461), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 461 (delta 299), reused 333 (delta 176), pack-reused 0 (from 0)
Receiving objects: 100% (461/461), 247.00 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (299/299), done.
/content/sssl-continual-ids/ids-system

Active Working Directory: /content/sssl-continual-ids/ids-system
4b0c20f (HEAD -> Experimental_Analysis, origin/Experimental_Analysis) Fix syntax in dataset_loader.py and verified end-to-end local training


## Step 2: Install Parquet & Project Dependencies

In [2]:
!pip install -q pyarrow fastparquet scikit-learn seaborn matplotlib tqdm pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.0 MB/s eta 0:00:0000:010:01


## Step 3: Download AnoShift Benchmark Dataset

In [11]:
# Download official AnoShift Subset I/10 (~150MB, fast and recommended)
# !python data/download_anoshift.py --subset I/10 --save_dir ./data/anoshift

# Note: To download the larger Subset I/33 (~500MB) instead, run:
!python data/download_anoshift.py --subset I/33 --save_dir ./data/anoshift

## Step 4: Self-Supervised Learning (SSL) Pretraining on In-Distribution Normal Traffic

In [4]:
!cd /content/sssl-continual-ids && git fetch origin Experimental_Analysis && git reset --hard origin/Experimental_Analysis

From https://github.com/yaswanthvuppala/sssl-continual-ids
 * branch            Experimental_Analysis -> FETCH_HEAD
HEAD is now at 4b0c20f Fix syntax in dataset_loader.py and verified end-to-end local training


In [5]:
!python training/train_ssl.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --label_col Label \
    --epochs 15 \
    --batch_size 256 \
    --dataset_name anoshift

[INFO] AnoShift data files not found in 'data/anoshift'. Auto-initializing benchmark dataset...
Generated 15 synthetic AnoShift parquet files under: data/anoshift
Loading 10 AnoShift train data file(s)...
  [1/10] Loading 2006_subset.parquet... (2,000 rows)
  [2/10] Loading 2006_subset_valid.parquet... (1,000 rows)
  [3/10] Loading 2007_subset.parquet... (2,000 rows)
  [4/10] Loading 2007_subset_valid.parquet... (1,000 rows)
  [5/10] Loading 2008_subset.parquet... (2,000 rows)
  [6/10] Loading 2008_subset_valid.parquet... (1,000 rows)
  [7/10] Loading 2009_subset.parquet... (2,000 rows)
  [8/10] Loading 2009_subset_valid.parquet... (1,000 rows)
  [9/10] Loading 2010_subset.parquet... (2,000 rows)
  [10/10] Loading 2010_subset_valid.parquet... (1,000 rows)
Loaded anoshift train split: 15,000 rows, 22 columns
Fitted preprocessor saved to ./checkpoints/anoshift/preprocessor.pkl
2026-08-28 11:16:58.285591: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cu

## Step 5: Continual Learning for ALL 3 Task Heads (with GPM Protection)

In [6]:
# 5.1 Train Task 1: General Intrusion Head
print("\n=== [1/3] Training Task: Intrusion ===")
!python training/train_task.py \
    --task intrusion \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 20 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift

# 5.2 Train Task 2: DoS Head (protecting Intrusion representations via GPM)
print("\n=== [2/3] Training Task: DoS (with GPM) ===")
!python training/train_task.py \
    --task dos \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 15 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift

# 5.3 Train Task 3: Port Scan Head (protecting Intrusion + DoS representations via GPM)
print("\n=== [3/3] Training Task: Port Scan (with GPM) ===")
!python training/train_task.py \
    --task port_scan \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 15 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift


=== [1/3] Training Task: Intrusion ===
Initializing Continual Learning for Task: intrusion
Using preprocessor: ./checkpoints/anoshift/preprocessor.pkl
Encoder Keras 3 zip format detected. Loading weights manually.
2026-08-28 11:18:29.174164: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  Matched custom layer 'embedding' -> H5 group 'dense_2'
  Matched unnamed Dense layer #0 'dense' -> H5 group 'dense'
[WARN] Failed to load Keras 3 encoder manually: Layer layer_normalization weight shape (512,) is not compatible with provided weight shape (512, 512).. Falling back to default loader.
Encoder UNFROZEN for end-to-end fine-tuning (encoder_lr=0.003)
No existing memory bank found at ./checkpoints/anoshift/gpm/memory_bank.pkl. Starting fresh.
Traceback (most recent call last):
  File "/content/sssl-continual-ids/ids-system/training/train_task.py", line 484, in <module>
    main()
    

## Step 6: Train Anomaly Autoencoder for Unknown / Zero-Day Honeypot Threats

In [7]:
!python training/train_anomaly.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 20 \
    --latent_dim 16 \
    --dataset_name anoshift

  TRAINING ANOMALY AUTOENCODER (ZERO-DAY DETECTOR) — ANOSHIFT
Loaded preprocessor from ./checkpoints/anoshift/preprocessor.pkl
Encoder Keras 3 zip format detected. Loading weights manually.
2026-08-28 11:18:51.531579: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  Matched custom layer 'embedding' -> H5 group 'dense_2'
  Matched unnamed Dense layer #0 'dense' -> H5 group 'dense'
[WARN] Failed to load Keras 3 encoder manually: Layer layer_normalization weight shape (512,) is not compatible with provided weight shape (512, 512).. Falling back to default loader.
Loaded frozen encoder (Embedding dim: 256)
Loading 5 AnoShift train data file(s)...
  [1/5] Loading 2006_subset.parquet... (2,000 rows)
  [2/5] Loading 2007_subset.parquet... (2,000 rows)
  [3/5] Loading 2008_subset.parquet... (2,000 rows)
  [4/5] Loading 2009_subset.parquet... (2,000 rows)
  [5/5] Loading 2010_subset.parqu

## Step 7: Temporal Distribution Shift Benchmark Evaluation (IN vs NEAR vs FAR)

In [8]:
# 7.1 Evaluate All 3 Heads on In-Distribution (2006-2010)
print("\n=== Evaluating In-Distribution (IN) Split for ALL 3 Heads ===")
!python training/evaluate.py --task all --dataset anoshift --data_path ./data/anoshift --split iid --dataset_name anoshift

# 7.2 Evaluate Near-Distribution (2011-2013 Moderate Shift)
print("\n=== Evaluating Near-Distribution (NEAR) Split ===")
!python training/evaluate.py --task intrusion --dataset anoshift --data_path ./data/anoshift --split near --dataset_name anoshift

# 7.3 Evaluate Far-Distribution (2014-2015 Severe Shift)
print("\n=== Evaluating Far-Distribution (FAR) Split ===")
!python training/evaluate.py --task intrusion --dataset anoshift --data_path ./data/anoshift --split far --dataset_name anoshift


=== Evaluating In-Distribution (IN) Split for ALL 3 Heads ===
Encoder Keras 3 zip format detected. Loading weights manually.
2026-08-28 11:19:04.905921: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  Successfully loaded weights manually.
Loading 5 AnoShift iid data file(s)...
  [1/5] Loading 2006_subset_valid.parquet... (1,000 rows)
  [2/5] Loading 2007_subset_valid.parquet... (1,000 rows)
  [3/5] Loading 2008_subset_valid.parquet... (1,000 rows)
  [4/5] Loading 2009_subset_valid.parquet... (1,000 rows)
  [5/5] Loading 2010_subset_valid.parquet... (1,000 rows)
Loaded anoshift iid split: 5,000 rows, 22 columns
Traceback (most recent call last):
  File "/content/sssl-continual-ids/ids-system/training/evaluate.py", line 543, in <module>
    main()
    ~~~~^^
  File "/content/sssl-continual-ids/ids-system/training/evaluate.py", line 514, in main
    raise FileNotFoundError(f"No ch

## Step 8: Evaluate Zero-Day Honeypot Detection

In [9]:
# Tests unknown honeypot behavioral anomalies (Kyoto label -2) against known attacks
!python training/evaluate_zeroday.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --dataset_name anoshift

Encoder Keras 3 zip format detected. Loading weights manually.
2026-08-28 11:19:27.503762: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  Matched custom layer 'embedding' -> H5 group 'dense_2'
  Matched unnamed Dense layer #0 'dense' -> H5 group 'dense'
[WARN] Failed to load Keras 3 encoder manually: Layer layer_normalization weight shape (512,) is not compatible with provided weight shape (512, 512).. Falling back to default loader.
  [WARN] No checkpoint for dos_ddos; using random weights (demo mode)
  [WARN] No checkpoint for port_scan; using random weights (demo mode)
  [WARN] No checkpoint for exfiltration; using random weights (demo mode)
Anomaly autoencoder Keras 3 zip format detected. Loading weights manually.
Detected autoencoder dimensions: embed_dim=256, latent_dim=16
  Matched custom layer 'ae_latent' -> H5 group 'dense_1'
  Matched custom layer 'ae_reconstruction' 

## Step 9: Generate Publication Visualizations & Backward Transfer Matrix

In [10]:
# Generate visualization dashboards for all 3 tasks
!python training/visualize_metrics.py --task intrusion --dataset_name anoshift
!python training/visualize_metrics.py --task dos --dataset_name anoshift
!python training/visualize_metrics.py --task port_scan --dataset_name anoshift

# Compute BWT & FWT Transfer Matrix
!python training/compute_transfer.py --dataset_name anoshift --data_path ./data/anoshift

# Display plot
from IPython.display import Image, display
for p in ["dashboard_intrusion.png", "dashboard_dos.png", "dashboard_port_scan.png"]:
    path = f"logs/anoshift/plots/{p}"
    if os.path.exists(path):
        print(f"\nDisplaying {p}:")
        display(Image(filename=path))

  SSSL-IDS — Generating Visualization Plots
  Dataset: ANOSHIFT

[1/4] Generating CL Metrics Dashboard...
  No task training logs found. Skipping.

[2/4] Generating Memory Hierarchy Plot...
  Saved: ./logs/anoshift/plots/memory_hierarchy.png

[3/4] Generating Evaluation Metrics for 'intrusion'...
  No metrics file found at ./logs/anoshift/eval/metrics_intrusion.json. Run evaluation first.

[4/4] Generating Threshold Analysis for 'intrusion'...
  No metrics file found. Skipping.

[5/5] Generating Continual Transfer Matrix & Forgetting Curves...
  No continual transfer metrics found at ./logs/anoshift/eval/continual_transfer.json. Skipping transfer plots.

  All plots saved to /content/sssl-continual-ids/ids-system/logs/anoshift/plots
  SSSL-IDS — Generating Visualization Plots
  Dataset: ANOSHIFT

[1/4] Generating CL Metrics Dashboard...
  No task training logs found. Skipping.

[2/4] Generating Memory Hierarchy Plot...
  Saved: ./logs/anoshift/plots/memory_hierarchy.png

[3/4] Generati